## 📊 Classificador de Planilhas
**Preencha os campos abaixo e clique no botão ▶ para rodar.**

> As planilhas serão classificadas automaticamente e enviadas
> para o e-mail informado. Nenhuma alteração manual é necessária.

In [ ]:
#@title ⚙️ Configurações — preencha os campos e clique em ▶
#@markdown ### 📂 Google Drive
PASTA_DRIVE = "TESTE" #@param {type:"string"}

#@markdown ### 📧 E-mail de envio
EMAIL_REMETENTE = "seuemail@gmail.com" #@param {type:"string"}
SENHA_APP = "xxxx xxxx xxxx xxxx" #@param {type:"string"}

#@markdown ### 📨 E-mail de destino
EMAIL_DESTINATARIO = "destinatario@email.com" #@param {type:"string"}

In [ ]:
import pandas as pd
import unicodedata
import openpyxl
import smtplib
import os
from email.message import EmailMessage
from google.colab import drive

print("=" * 50)
print("  CLASSIFICADOR DE PLANILHAS")
print("=" * 50)

# --- Conectar ao Drive ---
print("\n📂 Conectando ao Google Drive...")
drive.mount('/content/drive', force_remount=True)

caminho_pasta = f"/content/drive/MyDrive/{PASTA_DRIVE}"

if not os.path.exists(caminho_pasta):
    print(f"\n❌ ERRO: Pasta '{PASTA_DRIVE}' não encontrada no Drive.")
    print("Verifique o nome da pasta no campo de configuração.")
    raise SystemExit()

# --- Buscar planilhas nas subpastas ---
# Percorre tudo que estiver dentro de PASTA_DRIVE.
# Ignora PDFs, TXTs e qualquer .xlsx que não tenha
# a coluna "Discriminação".
print(f"\n🔍 Verificando conteúdo da pasta '{PASTA_DRIVE}'...")

arquivos_encontrados = []  # (caminho_completo, nome_subpasta, nome_arquivo)

for item in sorted(os.listdir(caminho_pasta)):
    caminho_item = os.path.join(caminho_pasta, item)

    if not os.path.isdir(caminho_item):
        continue  # Ignora arquivos soltos na raiz

    print(f"\n  📁 {item}/")

    for arquivo in os.listdir(caminho_item):
        if not arquivo.endswith('.xlsx'):
            print(f"     ⏭️  {arquivo} (ignorado)")
            continue

        caminho_arquivo = os.path.join(caminho_item, arquivo)

        try:
            df_teste = pd.read_excel(caminho_arquivo, nrows=0)
            if "Discriminação" not in df_teste.columns:
                print(f"     ⏭️  {arquivo} (sem coluna 'Discriminação' — ignorado)")
                continue
        except Exception:
            print(f"     ⚠️  {arquivo} (não foi possível ler — ignorado)")
            continue

        arquivos_encontrados.append((caminho_arquivo, item, arquivo))
        print(f"     ✅ {arquivo} (válido)")

if not arquivos_encontrados:
    print("\n❌ Nenhuma planilha válida encontrada.")
    raise SystemExit()

print(f"\n✅ {len(arquivos_encontrados)} planilha(s) válida(s) encontrada(s).")

# --- Funções de classificação ---
def limpar_texto(texto):
    if pd.isna(texto):
        return ""
    texto_str = str(texto).lower()
    return unicodedata.normalize('NFKD', texto_str).encode('ASCII', 'ignore').decode('utf-8')

def classificar_servico(descricao):
    desc = limpar_texto(descricao)

    termos_consulta = [
        'consulta medica', 'consultas medicas', 'coordenacao',
        'prestacao de servicos medico', 'prestacao ded servicos',
        'servicos medicos prestados', 'folha medica'
    ]
    termos_procedimento = [
        'procedimento', 'cirurgia', 'exame diagnostico',
        'servico hospitalar', 'ambulatorio', 'hospital', 'producao'
    ]

    tem_consulta     = any(termo in desc for termo in termos_consulta)
    tem_procedimento = any(termo in desc for termo in termos_procedimento)
    tem_atendimento  = 'atendimento' in desc

    if tem_atendimento or (tem_consulta and tem_procedimento):
        return 'ANALISAR'
    elif tem_consulta:
        return 'CONSULTA'
    elif tem_procedimento:
        return 'PROCEDIMENTO'
    else:
        return ''

# --- Processar cada planilha ---
arquivos_gerados = []
resumo_geral     = []
erros            = []

print("\n🔄 Processando planilhas...\n")

for i, (caminho_arquivo, nome_subpasta, nome_arquivo) in enumerate(arquivos_encontrados, 1):
    print(f"  [{i}/{len(arquivos_encontrados)}] {nome_subpasta}/{nome_arquivo}...", end=" ")

    try:
        wb_original  = openpyxl.load_workbook(caminho_arquivo)
        aba_original = wb_original.active
        nome_da_aba  = aba_original.title

        df = pd.read_excel(caminho_arquivo)
        df['Classificação'] = df['Discriminação'].apply(classificar_servico)

        nome_saida    = f"CLASSIFICADO_{nome_subpasta}_{nome_arquivo}"
        caminho_saida = f"/content/{nome_saida}"
        df.to_excel(caminho_saida, index=False, sheet_name=nome_da_aba, engine='openpyxl')

        wb_novo  = openpyxl.load_workbook(caminho_saida)
        aba_nova = wb_novo.active
        for letra, dimensao in aba_original.column_dimensions.items():
            aba_nova.column_dimensions[letra].width = dimensao.width
        wb_novo.save(caminho_saida)

        arquivos_gerados.append(caminho_saida)

        contagem     = df['Classificação'].value_counts().to_dict()
        vazias       = (df['Classificação'] == '').sum()
        linha_resumo = (
            f"• {nome_subpasta}/{nome_arquivo}: "
            f"CONSULTA={contagem.get('CONSULTA',0)} | "
            f"PROCEDIMENTO={contagem.get('PROCEDIMENTO',0)} | "
            f"ANALISAR={contagem.get('ANALISAR',0)}"
        )
        if vazias > 0:
            linha_resumo += f" | SEM CATEGORIA={vazias} ⚠️"

        resumo_geral.append(linha_resumo)
        print("✅")

    except Exception as e:
        print(f"❌ erro: {e}")
        erros.append(f"• {nome_subpasta}/{nome_arquivo}: {e}")

# --- Enviar e-mail ---
if not arquivos_gerados:
    print("\n❌ Nenhuma planilha foi processada. E-mail não enviado.")
else:
    print(f"\n📧 Enviando e-mail com {len(arquivos_gerados)} anexo(s)...")

    aviso_erros = ""
    if erros:
        aviso_erros = "\n\n⚠️ Arquivos com problema (não incluídos):\n" + "\n".join(erros)

    corpo = (
        f"Olá,\n\n"
        f"Seguem em anexo as {len(arquivos_gerados)} planilha(s) classificadas.\n\n"
        f"📊 Resumo por arquivo:\n"
        + "\n".join(resumo_geral)
        + aviso_erros +
        f"\n\nAtt,\nSistema de Classificação"
    )

    msg = EmailMessage()
    msg['Subject'] = f'Planilhas Classificadas — {len(arquivos_gerados)} arquivo(s) — Pasta: {PASTA_DRIVE}'
    msg['From']    = EMAIL_REMETENTE
    msg['To']      = EMAIL_DESTINATARIO
    msg.set_content(corpo)

    for caminho_saida in arquivos_gerados:
        with open(caminho_saida, 'rb') as f:
            msg.add_attachment(
                f.read(),
                maintype='application',
                subtype='vnd.openxmlformats-officedocument.spreadsheetml.sheet',
                filename=os.path.basename(caminho_saida)
            )

    try:
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
            smtp.login(EMAIL_REMETENTE, SENHA_APP)
            smtp.send_message(msg)
        print(f"✅ E-mail enviado para {EMAIL_DESTINATARIO}!")
    except Exception as e:
        print(f"❌ Erro ao enviar e-mail: {e}")
        print("Verifique se o e-mail e a senha de app estão corretos.")

# --- Resultado final ---
print("\n" + "=" * 50)
print(f"  CONCLUÍDO: {len(arquivos_gerados)}/{len(arquivos_encontrados)} planilhas processadas")
if erros:
    print(f"  ATENÇÃO: {len(erros)} arquivo(s) com problema")
print("=" * 50)